In [4]:
"""
add_error_category_to_bump.py
==============================
Safely adds normalized_error_category column to the BUMP CSV.
Uses csv module to reliably parse complex fields with embedded delimiters.
"""

import csv
import pandas as pd
from collections import Counter


# ─── CONFIG ───────────────────────────────────────────────────────────────────
BUMP_CSV   = "/Volumes/Rachna-HD/ConfigFiles/Candidate_BUMP_Instance_errorTypes.csv"
OUTPUT_CSV = "/Volumes/Rachna-HD/ConfigFiles/Candidate_BUMP_Instance_errorTypes_with_categories.csv"
# ──────────────────────────────────────────────────────────────────────────────

MAVEN_EXCLUDED = {'MojoFailureException', 'MojoExecutionException'}

CATEGORY_MAP: dict[str, str] = {
    'NullPointerException':         'NullPointerException',
    'ClassNotFoundException':       'ClassNotFoundException',
    'NoClassDefFoundError':         'NoClassDefFoundError',
    'NoSuchMethodError':            'NoSuchMethodError',
    'NoSuchMethodException':        'NoSuchMethodException',
    'NoSuchFieldError':             'NoSuchFieldError',
    'AbstractMethodError':          'AbstractMethodError',
    'IncompatibleClassChangeError': 'IncompatibleClassChangeError',
    'ClassCastException':           'ClassCastException',
    'IllegalAccessError':           'IllegalAccessError',
    'IllegalArgumentException':     'IllegalArgumentException',
    'VerifyError':                  'VerifyError',
    'LinkageError':                 'LinkageError',
    'ExceptionInInitializerError':  'ExceptionInInitializerError',
    'AssertionError':               'AssertionError',
    'AssertionFailedError':         'AssertionFailedError',
    'InvocationTargetException':    'InvocationTargetException',
}
OTHER = 'Other'


def categorize_exceptions(raw: str) -> str:
    if not raw or str(raw).strip() in ('', 'nan'):
        return OTHER
    cats_known = []
    has_other  = False
    for e in str(raw).split('|'):
        short = e.strip().split('.')[-1]
        if not short or any(ex in short for ex in MAVEN_EXCLUDED):
            continue
        cat = CATEGORY_MAP.get(short)
        if cat:
            if cat not in cats_known:
                cats_known.append(cat)
        else:
            has_other = True
    if not cats_known and not has_other:
        return OTHER
    return '|'.join(cats_known + ([OTHER] if has_other else []))


def detect_sep(path: str) -> str:
    """Detect separator from header line only — tab vs comma."""
    with open(path, encoding='utf-8') as f:
        header = f.readline()
    tabs   = header.count('\t')
    commas = header.count(',')
    sep = '\t' if tabs > commas else ','
    print(f"  Separator: {'TAB' if sep == chr(9) else 'COMMA'} (tabs={tabs}, commas={commas})")
    return sep


def main():
    sep = detect_sep(BUMP_CSV)

    # Read with python csv module — handles quoted fields with embedded delimiters
    df = pd.read_csv(
        BUMP_CSV,
        sep=sep,
        dtype=str,
        keep_default_na=False,   # don't convert empty strings to NaN
        quoting=csv.QUOTE_MINIMAL,
        engine='python',
    )
    print(f"  Loaded {len(df)} rows, {len(df.columns)} columns")
    print(f"  Columns: {list(df.columns)}")

    # Verify custom_id looks right
    print(f"  First 3 custom_ids: {df['custom_id'].head(3).tolist()}")

    # Add new column only — nothing else touched
    df['normalized_error_category'] = df['exception_types'].apply(categorize_exceptions)

    # Summary
    cat_counts: Counter = Counter()
    for val in df['normalized_error_category']:
        for c in str(val).split('|'):
            cat_counts[c.strip()] += 1
    print(f"\n  Category breakdown ({len(cat_counts)} categories):")
    for cat, cnt in sorted(cat_counts.items(), key=lambda x: -x[1]):
        print(f"    {cat:45s}  {cnt:3d} instances")

    # Write to new file with same separator and quoting
    df.to_csv(OUTPUT_CSV, sep=sep, index=False, quoting=csv.QUOTE_MINIMAL)
    print(f"\n  Saved → {OUTPUT_CSV}  ({len(df.columns)} columns, {len(df)} rows)")
    print(f"  Original {BUMP_CSV} is untouched.")


if __name__ == "__main__":
    main()

  Separator: COMMA (tabs=0, commas=30)
  Loaded 89 rows, 31 columns
  Columns: ['custom_id', 'clientGithubURL', 'clientProject', 'clientProjectOrganisation', 'breakingCommit', 'dependencyGroupID', 'dependencyArtifactID', 'previousVersion', 'newVersion', 'failureCategory', 'docker_image_breaking', 'execution_timestamp', 'execution_success', 'return_code', 'execution_time_seconds', 'tests_run', 'test_failures', 'test_errors', 'test_skipped', 'num_exceptions', 'num_failed_tests', 'num_compilation_errors', 'num_error_messages', 'exception_types', 'first_error_message', 'all_maven_errors', 'build_status', 'error_category', 'notes', 'log_file', 'parsed_errors_file']
  First 3 custom_ids: ['BBC01', 'BBC02', 'BBC03']

  Category breakdown (12 categories):
    Other                                           51 instances
    NoClassDefFoundError                            35 instances
    ClassNotFoundException                          30 instances
    ClassCastException                         